# Regularization in Machine Learning: Hands-on Implementation

## What This Notebook Covers
This notebook is the practical, code-driven counterpart to the **Regularization in Machine Learning README**. We will build an L2-regularized binary logistic regression model from scratch using NumPy, train it on a binary subset of the Iris dataset, and audit its loss components. Then, we will train Scikit-learn's optimized classifiers to evaluate the effects of regularization strength on accuracy and parameter magnitude.

## What You Will Accomplish
- Describe what regularization is and why it is needed to prevent overfitting.
- Explain the mathematical intuition behind L1 (Lasso) and L2 (Ridge) penalties.
- Implement an L2-regularized logistic regression training loop from scratch using NumPy.
- Compare your from-scratch model with Scikit-learn's optimized `LogisticRegression` classifier.
- Visualize the relationship between Scikit-learn's `C` hyperparameter, feature weights, and validation accuracy.

## Before You Start (Prerequisites)
- Comfort writing basic Python loops, methods, and list operations.
- Familiarity with NumPy array index queries and Pandas DataFrames.
- Basic understanding of classification evaluation tables.

## About the Dataset
This notebook uses the benchmark **Iris flower dataset**, containing 150 instances of flowers split evenly across three species: *setosa*, *versicolor*, and *virginica*. For every flower, we have four numeric measurements:
- Sepal length (cm)
- Sepal width (cm)
- Petal length (cm)
- Petal width (cm)

We load it directly using `sklearn.datasets.load_iris`. If you want to explore the dataset outside this notebook, it is also archived on Kaggle:
**https://www.kaggle.com/datasets/uciml/iris**

---

## 1. Setup & Workspace Preparation

### WHY?
Importing all libraries upfront with clean imports ensures that the workspace is configured correctly and sets the seeds for reproducible operations.

### HOW?
We import NumPy, Pandas, Matplotlib, Seaborn, and the necessary Scikit-learn helper tools, then apply custom plot styles.

In [ ]:
# Import NumPy for fast matrix arithmetic and manual MSE loss functions
import numpy as np

# Import Pandas to display data statistics within dataframes
import pandas as pd

# Import Matplotlib and Seaborn for plotting target class boundaries
import matplotlib.pyplot as plt
import seaborn as sns

# Import Iris dataset tools and models from sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seaborn style for clean grids
sns.set_style("whitegrid")

# Fix numpy seed for reproducibility
np.random.seed(42)

print("Libraries imported and workspace seed initialized.")

## 2. Dataset Loading & Exploration

### WHY?
Exploring the columns and distributions of the dataset before applying any transformations is key to identifying potential issues like class imbalance or scale variations.

### HOW?
We load the dataset using `load_iris()`, construct a Pandas DataFrame, and print statistical summaries.

In [ ]:
# Load iris dataset dictionary from sklearn
iris = load_iris()

# Wrap measurements inside a dataframe, applying original column names
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)

# Append target codes (0, 1, 2) to the dataframe
df['target'] = iris.target

# Peek at the first 5 records
print("First 5 rows of the dataset:")
display(df.head())

# Print dataframe properties to check coordinates and data types
print("\nDataset info:")
df.info()

# Print statistical characteristics (mean, std, min, max)
print("\nSummary statistics:")
display(df.describe())

print("\nFeatures:", iris.feature_names)
print("Target classes:", iris.target_names)

## 3. Preprocessing & Feature Scaling

### WHY?
Regularization penalizes weight sizes without knowing the scale of the features. To ensure that the penalty is applied evenly across features, we must standardize them to the same scale.

### HOW?
We audit the dataset for null values and duplicates, split it into train and test sets, and apply `StandardScaler` to normalize the feature dimensions.

In [ ]:
# Verify missing value counts per column
print("Missing values per column:")
print(df.isnull().sum())

# Count duplicate rows in the dataset
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Extract features X and targets y
X = df[iris.feature_names].values
y = df['target'].values

# Split into training (80%) and testing (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale the features so that all variables have a similar range.
# This ensures regularization penalizes each feature fairly.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTraining set shape:", X_train_scaled.shape)
print("Test set shape:", X_test_scaled.shape)

## 4. Part 3: L2 Regularized Logistic Regression From Scratch

### WHY?
Implementing regularized optimization from scratch using NumPy builds intuition for how the L2 penalty affects the gradient update step during training.

### HOW?
We define the mathematical loss functions, prediction probabilities, and update rules. To keep the math clear, we train on a binary classification subset (Species Setosa vs. others) and add the derivative of the L2 penalty term ($2 \lambda w$) to the weight updates.

In [ ]:
# Set up binary targets: Setosa (class 0) vs. all others
y_train_binary = (y_train == 0).astype(float)
y_test_binary = (y_test == 0).astype(float)

n_features = X_train_scaled.shape[1]

# Set numpy seed to make calculations deterministic
np.random.seed(42)

# Initialize weights and bias randomly (small values for stable training)
weights = np.random.randn(n_features) * 0.01
bias = 0.0

# Hyperparameters
learning_rate = 0.1
lambda_l2 = 0.1
n_epochs = 500

# Sigmoid activation function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Model prediction function
def predict_proba(X, w, b):
    z = np.dot(X, w) + b
    return sigmoid(z)

# Cost function incorporating L2 penalty
def compute_loss(X, y, w, b, lam):
    m = X.shape[0]
    y_pred = predict_proba(X, w, b)
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    
    # Binary cross-entropy loss
    data_loss = -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))
    
    # Regularization penalty (bias term is NOT regularized)
    l2_penalty = lam * np.sum(w ** 2)
    
    # Total regularized cost
    regularized_loss = data_loss + l2_penalty
    return data_loss, l2_penalty, regularized_loss

# Gradient descent optimization loop
for epoch in range(n_epochs):
    y_pred = predict_proba(X_train_scaled, weights, bias)
    error = y_pred - y_train_binary
    
    # Compute raw gradients for weights and bias
    grad_w = np.dot(X_train_scaled.T, error) / X_train_scaled.shape[0]
    grad_b = np.mean(error)
    
    # Add derivative of the L2 penalty term: d/dw [lambda * w^2] = 2 * lambda * w
    grad_w += 2 * lambda_l2 * weights
    
    # Update weights and bias
    weights -= learning_rate * grad_w
    bias -= learning_rate * grad_b

print("Training complete.")
print("Final weights:", np.round(weights, 4))
print("Final bias:", round(bias, 4))

## 5. Evaluating the From-Scratch Model

### WHY?
We evaluate predictions on the test set and log the breakdown of the final loss components to verify that our manual model generalizes correctly.

### HOW?
We run the test features through our prediction pipeline, compute loss terms, and evaluate test accuracy.

In [ ]:
# Compute probabilities and binary predictions on test data
test_probs = predict_proba(X_test_scaled, weights, bias)
test_preds = (test_probs >= 0.5).astype(int)

print("Sample predictions (probabilities):", np.round(test_probs[:10], 4))
print("Sample predicted classes:          ", test_preds[:10])
print("Sample true classes:               ", y_test_binary[:10].astype(int))

# Calculate loss breakdown
data_loss, l2_penalty, reg_loss = compute_loss(X_test_scaled, y_test_binary, weights, bias, lambda_l2)

print("\n--- Loss Breakdown (Test Set) ---")
print(f"Data loss (cross-entropy):  {data_loss:.4f}")
print(f"L2 penalty:                 {l2_penalty:.4f}")
print(f"Regularized loss (total):   {reg_loss:.4f}")

# Evaluate test accuracy
scratch_accuracy = np.mean(test_preds == y_test_binary)
print(f"\nFrom-scratch test accuracy: {scratch_accuracy:.4f}")

## 6. Part 4: Scikit-learn Classifier Comparison

### WHY?
Now, we compare the effects of strong vs. weak regularization on the full multi-class classification task using Scikit-learn's optimized solvers.

### HOW?
We train one model with strong regularization ($C=0.01$, large penalty) and one with weak regularization ($C=100$, small penalty). Then, we print accuracy, classification reports, and confusion matrices for both settings.

In [ ]:
# Train strongly regularized model (C=0.01 specifies strong penalty)
model_strong_reg = LogisticRegression(C=0.01, max_iter=1000, random_state=42)
model_strong_reg.fit(X_train_scaled, y_train)
preds_strong = model_strong_reg.predict(X_test_scaled)

# Train weakly regularized model (C=100 specifies weak penalty)
model_weak_reg = LogisticRegression(C=100, max_iter=1000, random_state=42)
model_weak_reg.fit(X_train_scaled, y_train)
preds_weak = model_weak_reg.predict(X_test_scaled)

print("===== LogisticRegression(C=0.01) — Strong Regularization =====")
print("Accuracy:", accuracy_score(y_test, preds_strong))
print("\nClassification Report:")
print(classification_report(y_test, preds_strong, target_names=iris.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds_strong))

print("\n===== LogisticRegression(C=100) — Weak Regularization =====")
print("Accuracy:", accuracy_score(y_test, preds_weak))
print("\nClassification Report:")
print(classification_report(y_test, preds_weak, target_names=iris.target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, preds_weak))

## 7. Visualizing Regularization Penalty Effects

### WHY?
Visualizing feature weight magnitudes for strong vs. weak regularization provides concrete, visual evidence of how the penalty term shrinks coefficients.

### HOW?
We extract the average absolute coefficient values for both models and plot them as a bar chart comparison.

In [ ]:
# Compute average absolute coefficient sizes across the features
mean_abs_weights_strong = np.mean(np.abs(model_strong_reg.coef_), axis=0)
mean_abs_weights_weak = np.mean(np.abs(model_weak_reg.coef_), axis=0)

x_pos = np.arange(len(iris.feature_names))
width = 0.35

# Plot the coefficient magnitudes comparison
plt.figure(figsize=(7, 4))
plt.bar(x_pos - width/2, mean_abs_weights_strong, width, label='C = 0.01 (Strong)', color='steelblue')
plt.bar(x_pos + width/2, mean_abs_weights_weak, width, label='C = 100 (Weak)', color='salmon')
plt.title('Mean Absolute Weight Magnitude per Feature')
plt.xlabel('Feature')
plt.ylabel('Mean |Coefficient|')
plt.xticks(x_pos, iris.feature_names, rotation=20, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Part 5: Hyperparameter Tuning Sweeps

### WHY?
To analyze the behavior of the regularization hyperparameter, we plot accuracy and weight magnitudes across a range of values for $C$, observing the transition from underfitting to overfitting.

### HOW?
We loop through a range of values for $C$, train a `LogisticRegression` model for each, track validation accuracy and average weight sizes, and compile them into a results table.

In [ ]:
# Define range of C values
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
results = []

# Iterate over parameters and log metrics
for c in C_values:
    model = LogisticRegression(C=c, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    avg_weight_magnitude = np.mean(np.abs(model.coef_))
    results.append((c, acc, avg_weight_magnitude))

# Wrap output within a dataframe table
results_df = pd.DataFrame(results, columns=['C', 'Test Accuracy', 'Mean |Weight|'])
print("Effect of C on Accuracy and Weight Magnitude:")
display(results_df)

# Part 6: Interview Corner

**Q1: What is Regularization?**
Regularization is a set of techniques that add a penalty term to a model's loss function based on the size of its parameters, discouraging overly complex models and improving generalization to unseen data.

**Q2: Why is Regularization needed?**
Without it, models — especially those with many parameters relative to the amount of training data — can fit noise in the training set, leading to high variance and poor performance on new data. Regularization constrains the parameter space to favor simpler, more generalizable solutions.

**Q3: L1 vs. L2 — what's the difference?**
L1 (Lasso) adds the sum of absolute weight values as a penalty and tends to drive some weights exactly to zero, effectively performing feature selection. L2 (Ridge) adds the sum of squared weight values and shrinks all weights toward zero smoothly, but rarely to exactly zero. L1 is preferred when feature selection or sparsity is desired; L2 is preferred when all features are believed to contribute and multicollinearity is a concern.

**Q4: What is Elastic Net?**
Elastic Net combines L1 and L2 penalties using a mixing parameter (often called `l1_ratio`), allowing the model to enjoy both the sparsity benefits of L1 and the stability benefits of L2 — particularly useful when features are correlated.

**Q5: What is Overfitting?**
Overfitting occurs when a model learns patterns specific to the training data (including noise and outliers) so closely that it performs well on training data but poorly on unseen test data.

**Q6: What is the Bias-Variance Tradeoff?**
Bias is the error from overly simplistic assumptions (underfitting), while variance is the error from sensitivity to small fluctuations in the training data (overfitting). Regularization increases bias slightly (by constraining the model) in exchange for a larger reduction in variance, often improving overall test performance — this balance is the bias-variance tradeoff.

# Key Takeaways

- Regularization adds a penalty on model weight magnitudes to the loss function, helping prevent overfitting and improve generalization.
- L1 (Lasso) encourages sparse models by driving some weights to exactly zero, while L2 (Ridge) shrinks all weights smoothly without eliminating them.
- Elastic Net blends L1 and L2 penalties to combine feature selection with stability on correlated features.
- In Scikit-learn, the `C` parameter is the inverse of regularization strength — smaller `C` means stronger regularization and typically smaller weight magnitudes.
- Regularization directly addresses the bias-variance tradeoff: it slightly increases bias but can substantially reduce variance, often improving performance on unseen data.